In [1]:
#####
#Import packages
import numpy as np
import matplotlib.pyplot as plt
from scipy import interpolate
from scipy import optimize
import math

#####
#Define a function to simulate the tissue curve with 1TCM
def cT_from_1TCM(k1,k2,times,input,unit_of_times):
    #here k1 and k2 are float numbers, times and input should be vectors of the same length, and unit_of_times is either 's' or 'min'
    #the units of k1 and k2 are assumed to be mL/min/mL and /min, respectively
    #the unit of the vector 'times' is specified by unit_of_times
    #the tissue curve cT will be in the same unit as the input
    #we first set the first value of cT as zero
    cT=[0]
    #then the rest of the cT values are obtained in an iterative loop
    for i in range(1,len(times)):
        if unit_of_times=='s':
            #we need the denominator 60 to since times are in seconds but K_1 and k_2 use /min
            cT.append(k1*(times[i]-times[i-1])/60*input[i-1]+
                (1-k2*(times[i]-times[i-1])/60)*cT[i-1])
        if unit_of_times=='min':
            #no denominator 60
            cT.append(k1*(times[i]-times[i-1])*input[i-1]+
                (1-k2*(times[i]-times[i-1]))*cT[i-1])
    return np.array(cT)

#####
#Define a function to simulate the tissue curve with a reversible 2TCM
def cT_from_reversible_2TCM(k1,k2,k3,k4,times,input,unit_of_times):
    #here k1,k2,k3,k4 are float numbers, and times and input should be vectors of the same length, unit_of_times is either 's' or 'min' 
    #the units of k1,k2,k3,k4 are assumed to be mL/min/mL, /min, /min, and /min
    #the unit of times is specified by unit_of_times
    #the first value of c1 and c2 are set as zero
    c1=[0]
    c2=[0]
    #then the rest of the values of c1 and c2 are obtained in an iterative loop
    for i in range(1,len(times)):
        if unit_of_times=='s':
            #we need the denominator 60 to since times are in seconds but k_is use /min
            c1.append(c1[i-1]+(times[i]-times[i-1])/60*(k1*input[i-1]-
                (k2+k3)*c1[i-1]+k4*c2[i-1]))
            c2.append(c2[i-1]+(times[i]-times[i-1])/60*(k3*c1[i-1]-
                k4*c2[i-1]))
        if unit_of_times=='min':
            #no denominator 60
            c1.append(c1[i-1]+(times[i]-times[i-1])*(k1*input[i-1]-
                (k2+k3)*c1[i-1]+k4*c2[i-1]))
            c2.append(c2[i-1]+(times[i]-times[i-1])*(k3*c1[i-1]-
                k4*c2[i-1]))
    c1=np.array(c1)
    c2=np.array(c2)
    cT=c1+c2   
    return c1,c2,cT

#####
#Define a function to simulate the tissue curve with a parallel 2TCM
def cT_from_parallel_2TCM(k1a,k2a,k1b,k2b,times,input,unit_of_times):
    #here k1,k2,k3,k4 are float numbers, and times and input should be vectors of the same length, unit_of_times is either 's' or 'min' 
    #the units of k1,k2,k3,k4 are assumed to be mL/min/mL, /min, /min, and /min
    #the unit of times is specified by unitOfTimes
    #the first value of c1 and c2 are set as zero
    c1=[0]
    c2=[0]
    #then the rest of the values of c1 and c2 are obtained in an iterative loop
    for i in range(1,len(times)):
        if unit_of_times=='s':
            #we need the denominator 60 to since times are in seconds but k_is use /min
            c1.append(c1[i-1]+(times[i]-times[i-1])/60*(k1a*input[i-1]-
                k2a*c1[i-1]))
            c2.append(c2[i-1]+(times[i]-times[i-1])/60*(k1b*input[i-1]-
                k2b*c2[i-1]))
        if unit_of_times=='min':
            #no denominator 60
            c1.append(c1[i-1]+(times[i]-times[i-1])*(k2a*input[i-1]-
                k2a*c1[i-1]))
            c2.append(c2[i-1]+(times[i]-times[i-1])*(k2b*input[i-1]-
                k2b*c2[i-1]))
    c1=np.array(c1)
    c2=np.array(c2)
    cT=c1+c2  
    return c1,c2,cT

#####
#Define an extended version of the interpolation function to avoid 'out-of-range' errors
def interpolate_extended(t,x,y):
    #if the point is on the interval, interpolate normally
    if x[0]<t<x[-1]:
         interpolate_1=interpolate.interp1d(x,y)
         return(interpolate_1(t))
    #if the point is before the interval, return the value at the first point of the interval
    elif t<=x[0]:
        return(y[0])
    #if the point after the interval, return the value at the last point of the interval
    else:
        return(y[-1])
    
#####
#Define a function to simulate the tissue curve with 1TCM with delay
def cT_from_1TCM_with_delay(k1,k2,times,input,delay,unit_of_times, unit_of_delay):
    #here k1,k2, and delay are float numbers, times and input should be vectors of the same length, and unit_of_times and unit_of_delay are 's' or 'min'
    #the units of k1 and k2 are assumed to be mL/min/mL and /min, respectively
    #the unit of times and delay are specified by unit_of_times and unit_of_delay
    #the first value of cT is set as zero
    cT=[0]
    #then the rest of the cT values are obtained in an iterative loop
    for i in range(1,len(times)):
        #let us define the value of input at time[i-1]-delay
        if unit_of_times==unit_of_delay:
            input_value=interpolate_extended(times[i-1]-delay,times,input)
        if unit_of_times=='min' and unit_of_delay=='s':
            input_value=interpolate_extended(times[i-1]-(1/60)*delay,times, input)
        if unit_of_times=='s' and unit_of_delay=='min':
            input_value=interpolate_extended(times[i-1]-60*delay,times,input)
        #let us append the current value of cT
        if unit_of_times=='s':
            cT.append(k1*(times[i]-times[i-1])/60*input_value
                +(1-k2*(times[i]-times[i-1])/60)*cT[i-1])
        if unit_of_times=='min':
            cT.append(k1*(times[i]-times[i-1])*input_value
                +(1-k2*(times[i]-times[i-1]))*cT[i-1])
    return np.array(cT)

#####
#Define a function for a typical dispersion correction that removes dispersion from the input
def remove_dispersion(times,input,tau,delay):
    #times and input are vectors of the same length, tau an delay are float numbers
    #the unit of times, tau, and delay is assumed to be in seconds
    corrected_input=[0]
    for i in range(1,len(times)):
        if delay==0:
            #if there is delay, the formula is very simple
            corrected_input.append(input[i]+tau*(input[i]-input[i-1])/(
                times[i]-times[i-1]))
        else:
            #if there is delay, we need to use interpolation to obtain the values at times[i]-delay and times[i-1]-delay
            delayed_input_i=interpolate_extended(times[i]-delay,times,
                input)
            delayed_input_i1=interpolate_extended(times[i-1]-delay,times,
                input)
            corrected_input.append(delayed_input_i+tau*(delayed_input_i
            -delayed_input_i1)/(times[i]-times[i-1]))
    return(np.array(corrected_input))

#####
#Define a function for an 'inverse' dispersion correction that adds dispersion to the input
def add_dispersion(times,input,tau,delay):
    #times and input are vectors of the same length, tau an delay are float numbers
    #the unit of times, tau, and delay is assumed to be in seconds
    #to remove dispersion, we use 1TCM
    #we need to choose K1 and k2 so that they correspond 1/tau if the time unit is the same
    #in other words, we fix K1=k2=1/(tau/60) since the unit of K1 and k2 is /min and tau is in seconds
    return(cT_from_1TCM_with_delay(1/(tau/60),1/(tau/60),times,input,
        delay,'s','s'))

#####
#Initiliaze some values for the model fitting:
#initialize the time points
time=[0,3,8,13,18,23,28,33,38,
    43,48,53,58,63,68,75,85,
    95,110,130,150,175,205,235,265]
times_interpolated = np.arange(0,266,5)
unit_of_times='s'
unit_of_delay='s'
#initilize the input
aorta=[0,124,886,5406,29009,84960,90750,78030,57405,
    42104,34094,27709,23290,22109,21168,18760,16693,
    15754,14843,14245,13780,13292,12567,12291,11974]
interpolate_input = interpolate.interp1d(time, aorta)
cA = interpolate_input(times_interpolated)
#initilize the tissue TAC 
# (the vector 'brain' is the mean values of the tracer concentration in 
# brain in the original time frames (in Bq/mL))
brain=[0,131,120,116,271,1984,4982,8035,10641,
    12466,13561,14294,14700,15073,15312,15330,15338,
    15308,15131,14855,14643,14123,13874,13473,13141]
interpolate_TAC = interpolate.interp1d(time, brain)
measured_TAC=interpolate_TAC(times_interpolated)

#####
#Define an error function computing the sum of squares between the measured TAC and the model curve cPET
def error_function(param):
    #param is a list of four numeric values
    #simulate cT and then cPET from the parameters in the param vector
    #use absolute value for param[0] and param[1] since k1 and k2 should be non-negative
    cT=cT_from_1TCM_with_delay(abs(param[0]),abs(param[1]),times_interpolated,
    cA,param[3],unit_of_times,unit_of_delay)
    #use inverse-logit for param[2] as V_a is between 0 and 1
    V_a=1/(1+math.exp(-param[2]))
    cPET=(1-V_a)*cT+V_a*cA
    #compute and return the sum of squares between the measured TAC and cPET
    sum_of_squares=np.sum((measured_TAC-cPET)**2)
    return sum_of_squares

#####
#Model fitting:
#choose initial values (here, param[2] is chosen so that V_a=0.1 after the inverse-logit transformation)
initial_values=[1,1,math.log(9),0]
#minimise the output of the error function
res=optimize.minimize(error_function,x0=initial_values)

In [3]:
######
# Import used packages
import numpy as np # numerical functions
from scipy import interpolate # interpolation
from scipy.stats import pearsonr # Pearson's correlation
import matplotlib.pyplot as plt # plotting

######
# Functions, define own functions
# Define discrete interval for all time points using Riemann integral
# tac=time activity curve with uniform time intervals (e.g. 1 s)
def cumulative_function(tac):
    res=np.ones(len(tac))
    res[0]=tac[0]
    for k in np.arange(1,len(res),1): 
        res[k]=tac[k]+res[k-1]
    return res

######
# Data, define arrays for data
# time_points: define time points here; start with 0
# TAC: define tissue activity here corresponding to time points
# TAC_input: define input function activity here corresponding to time points
# TAC_ref: define reference tissue activity here corresponding to time points

######
# Interpolation, interpolate data to have 1 Hz sampling rate
if 'time_points' in locals():
  interpolated_time_points=np.arange(0,time_points[len(time_points)-1]+1,1)
  if 'TAC' in locals():
    f=interpolate.interp1d(time_points,TAC)
    TAC_interpolated=f(interpolated_time_points)
  if 'TAC_input' in locals():
    f=interpolate.interp1d(time_points,TAC_input)
    TAC_input_interpolated=f(interpolated_time_points)
  if 'TAC_ref' in locals():
    f=interpolate.interp1d(time_points,TAC_ref)
    TAC_ref_interpolated=f(interpolated_time_points)

#####
# Calculate Riemann integral
# tac=time activity curve; first time point is expected to be 0
# time=mid point of time frames; first time point is expected to be 0
# end=frame number of the last frame to be included; between 1 and len(tac)
def Riemann_integral(tac,time,end):
  if len(tac)!=len(time):
    print("ERROR (Riemann_integral): input vectors have different lengths")
  # Define frame durations
  frames=np.zeros(len(tac)) # Initialise frames
  frames[0]=2*(time[1]-time[0]) # Find first value
  for k in range(1,len(tac)-2): # Find next values until the second to last
    frames[k]=2*(time[k+1]-time[k]-frames[k-1]/2)
  frames[len(tac)-2]=2*(time[len(tac)-1]-time[len(tac)-2]-frames[len(tac)-3]/2) # Find last value
  frames=frames[:(len(frames)-1)]
  # Find integral
  res=np.zeros(len(tac)) # Initialise integral
  for k in range(1,end+1):
    res[k]=tac[k]*frames[k-1]+res[k-1] # Add value of next frame
  res[k]=res[k]-tac[k]*frames[k-1]/2 # Last frame is corrected
  return round(res[k])

In [4]:
######
# Import used packages
import numpy as np
from scipy import interpolate
from scipy.optimize import nnls
import matplotlib.pyplot as plt

######
# Define spectral analysis related functions

######
# Define base function
# al=alpha (a numeric value)
# be=beta (a numeric value)
# time=time points (vector of numeric values; typically integers)
def base_function(al,be,time):
    return al*np.exp(-be*time)

######
# Define function for is convolution of a base function and input TAC
# al=alpha (a numeric value)
# be=beta (a numeric value)
# time=time points (vector of numeric values; typically integers)
# tac=time activity curve of input function with same length as time (vector of numeric values)
# NOTE: time and tac need to be of same length
def conv_comp(al,be,time,tac):
    return np.convolve(base_function(al,be,time),tac)

######
# Define operational function for spectral analysis
# al=alphas (a vector of non-negative numeric values)
# be=betas (a vector of non-negative numeric values)
# time=time points (vector of numeric values; typically integers)
# tac=time activity curve of input function with same length as time (vector of numeric values)
# NOTE: al and be need to be of same length; time and tac need to be of same length
def oper(al,be,time,tac):
    total_sum=0
    for k in range(len(al)):
        total_sum=total_sum+conv_comp(al[k],be[k],time,tac)
    return total_sum